In [1]:
!uv add paddleocr

Resolved 182 packages in 26ms
Checked 178 packages in 25ms


In [2]:
from pathlib import Path
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

### Normal English OCR

In [3]:
from paddleocr import PaddleOCR

ocr = PaddleOCR(
    lang="en",
    enable_mkldnn=False,
    use_doc_orientation_classify=True,
    use_doc_unwarping=True,
    use_textline_orientation=True,
)

c:\Users\IA Dev86\Nilesh\student_test_review_pipeline_POC\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\IA Dev86\Nilesh\student_test_review_pipeline_POC\.venv\Lib\site-packages\paddle\utils\cpp_extension\extension_utils.py:712: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
Creating model: ('PP-LCNet_x1_0_doc_ori', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\IA Dev86\.paddlex\official_models\PP-LCNet_x1_0_doc_ori`.
Creating model: ('UVDoc', None, None)
Model files already exist. Using cached files. To redownload, please delete the director

### Formula recognizer


In [4]:
from paddleocr import FormulaRecognitionPipeline

formula_pipeline_l = FormulaRecognitionPipeline(
    formula_recognition_model_name="PP-FormulaNet_plus-L",
    device="cpu",
    use_layout_detection=False,
    enable_mkldnn=False,
)

Creating model: ('PP-LCNet_x1_0_doc_ori', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\IA Dev86\.paddlex\official_models\PP-LCNet_x1_0_doc_ori`.
Creating model: ('UVDoc', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\IA Dev86\.paddlex\official_models\UVDoc`.
Creating model: ('PP-FormulaNet_plus-L', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\IA Dev86\.paddlex\official_models\PP-FormulaNet_plus-L`.


## Step 5 — Page Layout Detection

The formula recognition model works well when we provide it with an isolated
formula image. However, a real student-answer page may contain normal English
text, mathematical formulas, diagrams, tables, and other content.

We therefore first run a layout detection model over the complete page.

The layout detector identifies different regions of the page and provides:

- Region type/label
- Confidence score
- Bounding box coordinates

For our pipeline, the important regions are:

- `text` → send to PaddleOCR
- `formula` → send to FormulaNet
- `table` → handle separately later
- Other regions → preserve or handle later

This step acts as the router between the full page and the specialized OCR
models.

In [5]:
from paddleocr import LayoutDetection

## Step 6 — Initialize the Layout Detection Model

We use `PP-DocLayout_plus-L` to detect different regions on the page.

Unlike our FormulaNet pipeline, we do NOT disable layout detection here.
This model's entire purpose is to determine where text, formulas, tables,
and other regions are located.

We use CPU and disable MKL-DNN because the current environment is already
configured this way for the PaddleOCR models.

In [6]:
layout_pipeline = LayoutDetection(
    model_name="PP-DocLayout_plus-L",
    device="cpu",
    enable_mkldnn=False,
)

Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\IA Dev86\.paddlex\official_models\PP-DocLayout_plus-L`.


## Step 7 — Select a Full Student-Answer Page

The formula test above used an isolated formula image.

Now we test the layout detector using a complete student-answer page.

The page should contain normal English text and, ideally, at least one
mathematical expression so that we can verify whether the layout model
separates text and formula regions.

In [14]:
from pathlib import Path

page_path = Path("../test_data/maths_handwritten_notes.pdf")

if not page_path.exists():
    raise FileNotFoundError(
        f"File not found: {page_path.resolve()}"
    )

print("Page:", page_path.resolve())

Page: C:\Users\IA Dev86\Nilesh\student_test_review_pipeline_POC\src\test_data\maths_handwritten_notes.pdf


## Step 8 — Run Layout Detection

The complete page is passed to the layout detection model.

The model returns a list of detected regions. Each region contains:

- `label` — the detected region type
- `score` — confidence score
- `coordinate` — `[xmin, ymin, xmax, ymax]`

At this stage we are only inspecting the layout.
We are not running PaddleOCR or FormulaNet yet.

In [15]:
layout_result = layout_pipeline.predict(
    str(page_path),
    batch_size=1,
    layout_nms=True,
)

## Step 9 — Inspect Detected Regions

Before connecting the OCR models, we inspect what the layout detector
actually found.

This is an important debugging step because the rest of the pipeline
depends on correct region detection.

For example, we ideally want something similar to:

    text       → English paragraph
    formula    → Mathematical equation
    text       → English explanation
    formula    → Another equation

The coordinates tell us where each region exists on the original page.

In [16]:
for page_no, res in enumerate(layout_result, start=1):

    data = res.json["res"]

    print(f"\n========== PAGE {page_no} ==========")

    boxes = data.get("boxes", [])

    print(f"Detected regions: {len(boxes)}")

    for i, box in enumerate(boxes, start=1):

        print(f"\nRegion {i}")
        print(f"  Label      : {box['label']}")
        print(f"  Confidence : {box['score']:.3f}")
        print(f"  Coordinate : {box['coordinate']}")


========== PAGE 1 ==========
Detected regions: 1

Region 1
  Label      : text
  Confidence : 0.606
  Coordinate : [193.6173553466797, 1159.004150390625, 1071.5740966796875, 1312.7257080078125]

========== PAGE 2 ==========
Detected regions: 1

Region 1
  Label      : text
  Confidence : 0.515
  Coordinate : [223.96884155273438, 1349.677978515625, 802.2190551757812, 1474.615966796875]

========== PAGE 3 ==========
Detected regions: 2

Region 1
  Label      : formula
  Confidence : 0.582
  Coordinate : [288.1087951660156, 1185.5714111328125, 593.3536376953125, 1258.274658203125]

Region 2
  Label      : text
  Confidence : 0.518
  Coordinate : [182.171630859375, 307.98681640625, 1062.5426025390625, 1365.56103515625]

========== PAGE 4 ==========
Detected regions: 2

Region 1
  Label      : header
  Confidence : 0.606
  Coordinate : [706.2507934570312, 161.77540588378906, 874.9331665039062, 256.49078369140625]

Region 2
  Label      : text
  Confidence : 0.584
  Coordinate : [205.553359

## Step 10 — Visualize the Detected Layout

The layout detector can generate a visualization of the detected regions.

This allows us to visually verify whether:

- English text was detected correctly
- Mathematical formulas were detected correctly
- The bounding boxes cover the complete content
- Formula regions are separated from surrounding text

We should verify this visually before building the OCR routing logic.

In [17]:
output_dir = Path("../test_data/layout_output")
output_dir.mkdir(parents=True, exist_ok=True)

for res in layout_result:
    res.save_to_img(
        save_path=str(output_dir)
    )

print("Saved layout visualization to:")
print(output_dir.resolve())

Saved layout visualization to:
C:\Users\IA Dev86\Nilesh\student_test_review_pipeline_POC\src\test_data\layout_output


## Step 11 — Extract Text and Formula Regions

For the next stage we only care about regions that can be processed by
our two OCR engines.

Text regions will eventually be processed by PaddleOCR.

Formula regions will eventually be processed by FormulaNet.

At this stage we only collect the detected regions. We do not perform
OCR yet.

In [18]:
regions = []

for page_no, res in enumerate(layout_result, start=1):

    data = res.json["res"]

    for box in data.get("boxes", []):

        label = box["label"]
        score = float(box["score"])
        x1, y1, x2, y2 = box["coordinate"]

        if label in {"text", "formula"}:

            regions.append({
                "page": page_no,
                "type": label,
                "score": score,
                "bbox": [
                    int(x1),
                    int(y1),
                    int(x2),
                    int(y2),
                ],
            })

for region in regions:
    print(region)

{'page': 1, 'type': 'text', 'score': 0.6064968705177307, 'bbox': [193, 1159, 1071, 1312]}
{'page': 2, 'type': 'text', 'score': 0.5148417353630066, 'bbox': [223, 1349, 802, 1474]}
{'page': 3, 'type': 'formula', 'score': 0.5817606449127197, 'bbox': [288, 1185, 593, 1258]}
{'page': 3, 'type': 'text', 'score': 0.5176133513450623, 'bbox': [182, 307, 1062, 1365]}
{'page': 4, 'type': 'text', 'score': 0.5836367607116699, 'bbox': [205, 424, 1096, 657]}


## Step — Display OCR Results in a Readable Format

This cell converts the raw PaddleOCR response into a human-readable format.

Instead of printing the complete PaddleOCR response dictionary, we display
each detected text region separately with:

- Region number
- Recognized text
- Confidence score
- Bounding box

This makes it easier to inspect OCR quality before sending the extracted
content to the next stage of the pipeline.

In [19]:

# ============================================================
# GRADEFORGE - CORRECTED OCR + FORMULA EXTRACTION
# ============================================================

from pathlib import Path
from collections import Counter
from pprint import pprint
from PIL import Image
import fitz

# ------------------------------------------------------------
# 1. Render PDF pages to images
# ------------------------------------------------------------

render_dir = Path("../test_data/rendered_pages")
render_dir.mkdir(parents=True, exist_ok=True)

pdf_document = fitz.open(page_path)
page_images = {}

for page_index in range(len(pdf_document)):
    page = pdf_document[page_index]

    # 2x resolution for better OCR/formula recognition.
    matrix = fitz.Matrix(2, 2)
    pix = page.get_pixmap(matrix=matrix, alpha=False)

    image_path = render_dir / f"page_{page_index + 1}.png"
    pix.save(str(image_path))
    page_images[page_index + 1] = image_path

pdf_document.close()

print("Rendered pages:")
for page_no, image_path in page_images.items():
    print(f"  Page {page_no}: {image_path}")


# ------------------------------------------------------------
# 2. Inspect all labels detected by the layout model
# ------------------------------------------------------------

labels = []

for page_no, res in enumerate(layout_result, start=1):
    data = res.json

    if isinstance(data, dict) and "res" in data:
        data = data["res"]

    for box in data.get("boxes", []):
        label = box.get("label")
        if label:
            labels.append(label)

print("\nDetected layout labels:")
for label, count in Counter(labels).most_common():
    print(f"  {label:30} {count}")


# ------------------------------------------------------------
# 3. Normalize regions from the layout detector
# ------------------------------------------------------------

def normalize_regions(layout_results):
    normalized = []

    for page_no, res in enumerate(layout_results, start=1):
        data = res.json

        if isinstance(data, dict) and "res" in data:
            data = data["res"]

        for box in data.get("boxes", []):
            label = str(box.get("label", "")).lower()
            score = float(box.get("score", 0.0))

            raw_bbox = box.get("coordinate") or box.get("bbox")

            if raw_bbox is None:
                continue

            # Keep only regions useful for our OCR POC.
            if label not in {
                "text",
                "formula",
                "title",
                "paragraph_title",
                "figure",
                "table",
            }:
                continue

            normalized.append({
                "page": page_no,
                "type": label,
                "score": score,
                "bbox": raw_bbox,
            })

    return normalized


regions = normalize_regions(layout_result)

print(f"\nTotal OCR/layout regions: {len(regions)}")

for i, region in enumerate(regions, start=1):
    print(
        f"Region {i}: "
        f"page={region['page']} "
        f"type={region['type']} "
        f"score={region['score']:.4f} "
        f"bbox={region['bbox']}"
    )


# ------------------------------------------------------------
# 4. Process text/formula regions
# ------------------------------------------------------------

ocr_results = []

for index, region in enumerate(regions, start=1):

    page_no = region["page"]
    region_type = region["type"]
    bbox = region["bbox"]

    image_path = page_images.get(page_no)

    if image_path is None:
        continue

    image = Image.open(image_path)

    # The PDF was rendered at 2x.
    scale = 2

    x1, y1, x2, y2 = [float(v) for v in bbox]

    crop_box = (
        max(0, int(x1 * scale)),
        max(0, int(y1 * scale)),
        min(image.width, int(x2 * scale)),
        min(image.height, int(y2 * scale)),
    )

    if crop_box[2] <= crop_box[0] or crop_box[3] <= crop_box[1]:
        continue

    crop = image.crop(crop_box)

    crop_dir = Path("../test_data/region_crops")
    crop_dir.mkdir(parents=True, exist_ok=True)

    crop_path = (
        crop_dir / f"page_{page_no}_{region_type}_{index}.png"
    )

    crop.save(crop_path)

    item = {
        "page": page_no,
        "type": region_type,
        "layout_score": region["score"],
        "bbox": bbox,
        "crop_path": str(crop_path),
    }

    # --------------------------------------------------------
    # TEXT OCR
    # --------------------------------------------------------

    if region_type in {
        "text",
        "title",
        "paragraph_title",
    }:

        text_result = ocr.predict(str(crop_path))

        if text_result:
            data = text_result[0].json

            if isinstance(data, dict) and "res" in data:
                data = data["res"]

            texts = data.get("rec_texts", [])
            scores = data.get("rec_scores", [])

            item["text"] = " ".join(
                str(text).strip()
                for text in texts
                if str(text).strip()
            )

            item["ocr_scores"] = [
                float(score) for score in scores
            ]

    # --------------------------------------------------------
    # FORMULA RECOGNITION
    # --------------------------------------------------------

    elif region_type == "formula":

        formula_result = formula_pipeline_l.predict(
            str(crop_path)
        )

        if formula_result:
            formula_data = formula_result[0].json

            if isinstance(formula_data, dict) and "res" in formula_data:
                formula_data = formula_data["res"]

            item["formula_result"] = formula_data
        else:
            item["formula_result"] = None

    ocr_results.append(item)


print(f"\nProcessed regions: {len(ocr_results)}")


# ------------------------------------------------------------
# 5. Display final structured result
# ------------------------------------------------------------

def display_ocr_result(results):
    print("\n" + "=" * 90)
    print("                    GRADEFORGE OCR RESULT")
    print("=" * 90)

    current_page = None

    for index, item in enumerate(results, start=1):

        page_no = item["page"]

        if page_no != current_page:
            print("\n" + "=" * 90)
            print(f"PAGE {page_no}")
            print("=" * 90)
            current_page = page_no

        print(f"\n[{index}] {item['type'].upper()}")
        print(f"  Layout confidence : {item['layout_score']:.4f}")
        print(f"  Bounding box      : {item['bbox']}")
        print(f"  Crop              : {item['crop_path']}")

        if item["type"] in {
            "text",
            "title",
            "paragraph_title",
        }:
            print(f"  Text              : {item.get('text', '')}")

            scores = item.get("ocr_scores", [])

            if scores:
                print(
                    "  OCR confidence    : "
                    + ", ".join(
                        f"{score:.4f}" for score in scores
                    )
                )

        elif item["type"] == "formula":
            print("  Formula result:")

            formula_data = item.get("formula_result")

            if formula_data:
                pprint(formula_data, sort_dicts=False)
            else:
                print("    No formula result returned.")


# IMPORTANT:
# We call the variable that actually exists: `ocr_results`.
display_ocr_result(ocr_results)


FzErrorSystem: code=2: cannot remove file '..\test_data\rendered_pages\page_1.png': Permission denied

In [ ]:
# ============================================================
# FINAL FORMATTED OUTPUT
# ============================================================

from collections import defaultdict
from pprint import pprint

def display_formatted_result(results):
    pages = defaultdict(list)

    # Group everything page-wise
    for item in results:
        pages[item["page"]].append(item)

    print("\n")
    print("=" * 100)
    print("                         GRADEFORGE")
    print("                  DOCUMENT EXTRACTION RESULT")
    print("=" * 100)

    print(f"\nTotal Pages : {len(pages)}")
    print(f"Total Regions: {len(results)}")

    for page_no in sorted(pages):

        print("\n\n")
        print("=" * 100)
        print(f"PAGE {page_no}")
        print("=" * 100)

        page_items = pages[page_no]

        for item_no, item in enumerate(page_items, start=1):

            region_type = item["type"]

            print("\n" + "-" * 90)

            # ------------------------------------------------
            # TEXT
            # ------------------------------------------------

            if region_type in {
                "text",
                "title",
                "paragraph_title",
            }:

                text = item.get("text", "").strip()

                if not text:
                    continue

                print(f"[{item_no}] {region_type.upper()}")

                print("\n" + text)

            # ------------------------------------------------
            # FORMULA
            # ------------------------------------------------

            elif region_type == "formula":

                print(f"[{item_no}] FORMULA")

                formula_data = item.get("formula_result")

                if not formula_data:
                    print("\n[Formula detected but no LaTeX returned]")
                    continue

                # PaddleOCR formula result
                formula_found = False

                if isinstance(formula_data, dict):

                    # Common PaddleOCR structure
                    formula_list = formula_data.get(
                        "formula_res_list",
                        []
                    )

                    if formula_list:

                        for formula in formula_list:

                            latex = formula.get(
                                "rec_formula"
                            )

                            if latex:
                                formula_found = True

                                print("\nLaTeX:")
                                print(latex)

                    # Sometimes result may directly contain
                    # rec_formula
                    latex = formula_data.get(
                        "rec_formula"
                    )

                    if latex:
                        formula_found = True

                        print("\nLaTeX:")
                        print(latex)

                if not formula_found:
                    print("\nRaw formula result:")
                    pprint(
                        formula_data,
                        sort_dicts=False
                    )

            # ------------------------------------------------
            # OTHER
            # ------------------------------------------------

            else:

                print(f"[{item_no}] {region_type.upper()}")

                if item.get("text"):
                    print("\n" + item["text"])


# IMPORTANT:
# Use the variable created by the OCR processing cell.
display_formatted_result(ocr_results)



                         GRADEFORGE
                  DOCUMENT EXTRACTION RESULT

Total Pages : 1
Total Regions: 9



PAGE 1

------------------------------------------------------------------------------------------

------------------------------------------------------------------------------------------
[2] TEXT

nplexity of binary search, and why?

------------------------------------------------------------------------------------------

------------------------------------------------------------------------------------------
[4] TEXT

PDDDD

------------------------------------------------------------------------------------------

------------------------------------------------------------------------------------------

------------------------------------------------------------------------------------------
[7] TEXT

SASUOOSAPUESSBSBBSS3

------------------------------------------------------------------------------------------
[8] PARAGRAPH_TITLE

er: A database index is